<a href="https://www.kaggle.com/code/atrbyg24/predicting-optimal-fertilizers?scriptVersionId=243255955" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

In [ ]:
train = pd.read_csv('/kaggle/input/playground-series-s5e6/train.csv',index_col='id')
test = pd.read_csv('/kaggle/input/playground-series-s5e6/test.csv')

In [ ]:
train.head()

In [ ]:
train.info()

In [ ]:
train.describe()

In [ ]:
cat_cols = ['Soil Type','Crop Type','Fertilizer Name']
num_cols = ['Temparature','Humidity','Moisture','Nitrogen','Potassium','Phosphorous']

In [ ]:
for col in cat_cols:
    print(train[col].value_counts())

In [ ]:
for col in num_cols:
    plt.figure()
    ax = sns.histplot(data = train,x=col,hue='Fertilizer Name',multiple='stack')
    plt.title(f'Histogram of {col}')
    sns.move_legend(ax, loc='upper left', bbox_to_anchor=(1.02, 1))
    plt.show()
    plt.clf()

In [ ]:
y = train['Fertilizer Name'].copy()
X = train.drop('Fertilizer Name', axis=1).copy()

X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.7, shuffle=True, random_state=17)

In [ ]:
nominal_transformer = Pipeline(steps=[
    ('onehot', OneHotEncoder(sparse=False))
])

preprocessor = ColumnTransformer(transformers=[
    ('nominal', nominal_transformer, ['Soil Type', 'Crop Type'])
], remainder='passthrough')

model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('scaler', StandardScaler()),
    ('classifier', RandomForestClassifier())
])

In [ ]:
model.fit(X_train, y_train)

In [ ]:
y_pred = model.predict(X_test)

In [ ]:
probabilities = model.predict_proba(X_test)
print("\nPredicted probabilities on test set (first 5 samples):\n", probabilities[:5])


print("\nModel Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

classes = model.named_steps['classifier'].classes_ 

results = []
for i, prob_array in enumerate(probabilities[:5]): 
    sorted_indices = np.argsort(prob_array)[::-1]
    top3 = []
    for j in range(min(3, len(classes))):
        class_label = classes[sorted_indices[j]]
        top3.append(class_label)
    results.append(" ".join(top3))
print(results)

In [ ]:
test_preds = model.predict_proba(test)
top_3_fertilizers = []
for i, pred in enumerate(test_preds):
    top_indices = np.argsort(pred)[::-1]  
    top3 = []
    for j in range(min(3, len(classes))):
        class_label = classes[sorted_indices[j]]
        top3.append(class_label)
    top_3_fertilizers.append(" ".join(top3))

submission = pd.DataFrame({'id': test['id'], 'Fertilizer Name': top_3_fertilizers})

submission.to_csv('submission.csv', index=False)

print("Submission file created successfully!")
submission.head()